# House Data Preprocessing

This notebook preprocesses the cleaned house dataset and exports model-ready features.

## Flow
1. Setup and load cleaned house data
2. Parse address into structured location parts
3. Build preprocessing pipeline (Yes/No binary + OHE + numeric passthrough)
4. Export preprocessed matrix
5. Export model feature block and OOF location encoded features

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('Environment setup complete.')

Environment setup complete.


In [2]:
df = pd.read_csv('../../data/house/satilir_properties_house_cleaned.csv')
print('Dataset shape:', df.shape)
print('Total missing values:', int(df.isna().sum().sum()))

df.head()

Dataset shape: (1173, 25)
Total missing values: 0


,price,rooms,area_m2,land_area_sot,floor,has_document,address,avtodayanacaq,balkon,duzelme,esyali,hovuz,internet,isiq,kabel_tv,kombi,kondisioner,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirli
0,125000,3.0,100.000000,1.7,1.0,Yes,Xırdalan,No,Yes,No,No,No,Yes,Yes,Yes,Yes,No,No,No,Yes,Yes,No,Yes,No,Yes
1,145000,3.0,110.000000,2.2,1.0,Yes,Xırdalan,Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,No,Yes
2,35000,1.0,60.000000,0.6,1.0,Yes,"Bakı, Abşeron, Mehdiabad",No,No,No,No,No,Yes,Yes,Yes,Yes,No,No,No,No,Yes,Yes,Yes,No,Yes
3,279000,6.0,250.000000,2.5,3.0,Yes,"Bakı, Binəqədi, Biləcəri qəs., metro Avtovağzal",Yes,Yes,No,Yes,No,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes
4,75000,6.0,204.700091,4.0,2.0,Yes,Xızı,No,No,No,No,No,No,Yes,No,No,No,No,No,No,Yes,Yes,Yes,No,No


In [3]:
yes_no_cols = []
for col in df.columns:
    non_null = df[col].dropna().astype(str).str.strip().str.lower()
    if len(non_null) and set(non_null.unique()).issubset({"yes", "no"}):
        yes_no_cols.append(col)

print("Detected Yes/No columns:", yes_no_cols)

Detected Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli']


In [4]:
print(f"Before separating address: {df.shape[1]} columns")

Before separating address: 25 columns


In [5]:
candidate_cols = ["address"]
address_col = next((c for c in candidate_cols if c in df.columns), None)

if address_col is None:
    raise ValueError(f"No address column found. Available columns: {list(df.columns)}")

address_parts = ["address_part_1", "address_part_2", "address_part_3", "address_part_4"]

parts = (
    df["address"]
    .fillna("")
    .astype(str)
    .str.split(",", expand=True)
    .iloc[:, :4]
)
parts.columns = address_parts[:parts.shape[1]]

for col in address_parts:
    if col not in parts.columns:
        parts[col] = ""


df = pd.concat([df, parts[address_parts]], axis=1)

df[address_parts].head()

,address_part_1,address_part_2,address_part_3,address_part_4
0,Xırdalan,None,None,None
1,Xırdalan,None,None,None
2,Bakı,Abşeron,Mehdiabad,None
3,Bakı,Binəqədi,Biləcəri qəs.,metro Avtovağzal
4,Xızı,None,None,None


In [6]:
print(f"After separating address: {df.shape[1]} columns")

After separating address: 29 columns


In [7]:
df["address"]

0                                              Xırdalan
1                                              Xırdalan
2                              Bakı, Abşeron, Mehdiabad
3       Bakı, Binəqədi, Biləcəri qəs., metro Avtovağzal
4                                                  Xızı
                             ...                       
1168                             Bakı, Xəzər, Binə qəs.
1169                             Bakı, Xəzər, Binə qəs.
1170                             Bakı, Abşeron, Masazır
1171                               Bakı, Abşeron, Saray
1172                             Bakı, Abşeron, Novxanı
Name: address, Length: 1173, dtype: object

In [8]:
df.drop(columns=["address"], inplace=True)

In [9]:
df[address_parts].isnull().sum()/len(df)

address_part_1    0.000000
address_part_2    0.138960
address_part_3    0.180733
address_part_4    0.905371
dtype: float64

In [10]:
df["address_part_4"].unique()

array([None, ' metro Avtovağzal', ' metro Koroğlu', ' metro Azadlıq',
       ' metro Xətai', ' metro İnşaatçılar', ' metro Gənclik',
       ' metro Əhmədli', ' metro Xocaəsən', ' metro Sahil',
       ' metro Neftçilər', ' metro Nizami', ' metro 8 Noyabr',
       ' metro İçərişəhər', ' metro Nəriman Nərimanov', ' metro Nəsimi',
       ' metro Elmlər akademiyası', ' metro Həzi Aslanov'], dtype=object)

In [11]:
df.drop(columns=["address_part_4"], inplace=True)

In [12]:
df["address_part_3"].unique()

array([None, ' Mehdiabad', ' Biləcəri qəs.', ' Mərdəkan', ' Binə qəs.',
       ' Nardaran qəs.', ' Şüvəlan', ' metro İnşaatçılar', ' Masazır',
       ' Zabrat qəs.', ' Novxanı', ' metro Nəsimi', ' Bilgəh qəs.',
       ' Məhəmmədli', ' Lökbatan qəs.', ' Goradil', ' Savalan qəs.',
       ' metro Qara Qarayev', ' Bakıxanov qəs.', ' Türkan', ' Buzovna',
       ' Binəqədi qəs.', ' Maştağa qəs.', ' Badamdar qəs.', ' Zığ qəs.',
       ' Saray', ' metro Xalqlar dostluğu', ' Qala', ' metro Nizami',
       ' Ağ şəhər', ' Kürdəxanı qəs.', ' metro Həzi Aslanov',
       ' Hövsan qəs.', ' Yasamal qəs.', ' Yeni Ramana', ' Ramana qəs.',
       ' Böyükşor qəs.', ' 8-ci mikrorayon', ' Bülbülə qəs.', ' Əhmədli',
       ' Xocəsən qəs.', ' Pirşağı qəs.', ' Qaraçuxur qəs.',
       ' Rəsulzadə qəs.', ' Balaxanı qəs.', ' Şağan', ' Qobu',
       ' 28 may qəs.', ' Dübəndi', ' Sabunçu qəs.', ' Kubinka',
       ' Hökməli', ' Papanin', ' 20-ci sahə', ' metro Gənclik',
       ' Yeni Suraxanı qəs.', ' Montin qəs.', 

In [13]:
df.drop(columns=["address_part_3"], inplace=True)

In [14]:
df["address_part_2"].unique()

array([None, ' Abşeron', ' Binəqədi', ' Xəzər', ' Sabunçu', ' Yasamal',
       ' Yaşıl dərə', ' Qaradağ', ' St.Sumqayıt', ' Xəzər bağları',
       ' Qurd dərəsi', ' Nizami', ' Səbail', ' Suraxanı', ' Xətai',
       ' Nərimanov', ' Kotec', ' Corat bağları', ' Birləşmiş məhəllə',
       ' Nəsimi', ' 21-ci mikrorayon'], dtype=object)

In [15]:
df["address_part_1"].unique()

array(['Xırdalan', 'Bakı', 'Xızı', 'Sumqayıt', 'Zaqatala', 'Qəbələ',
       'Şəki', 'Quba', 'Gəncə', 'Tovuz', 'Xaçmaz', 'Masallı', 'Lənkəran',
       'Sabirabad', 'Şamaxı', 'İsmayıllı', 'Lerik', 'Astara', 'Siyazən'],
      dtype=object)

In [16]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Total columns: {df.shape[1]}")
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Yes No columns ({len(yes_no_cols)}): {yes_no_cols}")
print(f"Address parts (2): address_part_1, address_part_2")

schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
})
schema_df

Total columns: 26
Numeric (5): ['price', 'rooms', 'area_m2', 'land_area_sot', 'floor']
Categorical (21): ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli', 'address_part_1', 'address_part_2']
Yes No columns (19): ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli']
Address parts (2): address_part_1, address_part_2


,column,dtype
0,price,int64
1,rooms,float64
2,area_m2,float64
3,land_area_sot,float64
4,floor,float64
5,has_document,object
6,avtodayanacaq,object
7,balkon,object
8,duzelme,object
9,esyali,object


Encoding pipeline

In [18]:
class YesNoBinaryEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.feature_names_in_ = X_df.columns.to_numpy()
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.feature_names_in_)
        out = pd.DataFrame(index=X_df.index)
        for col in self.feature_names_in_:
            out[col] = (
                X_df[col]
                .astype('string')
                .str.strip()
                .str.lower()
                .map({'yes': 1, 'no': 0})
                .astype(float)
            )
        return out.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return np.asarray(input_features, dtype=object)


y = df["price"].copy()
X_df = df.drop(columns=["price"], errors="ignore").copy()

yes_no_cols_model = [c for c in yes_no_cols if c in X_df.columns]

categorical_cols = X_df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
categorical_base = [c for c in categorical_cols if c not in yes_no_cols_model]


cardinality = X_df[categorical_base].nunique(dropna=False).sort_values(ascending=False)
low_card_cols = [c for c in categorical_base if X_df[c].nunique(dropna=False) <= 15]
mid_card_cols = [c for c in categorical_base if 15 < X_df[c].nunique(dropna=False) <= 80]


numeric_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()


print("Yes/No columns:", yes_no_cols_model)
print("Low-cardinality categorical:", low_card_cols)
print("Mid-cardinality categorical:", mid_card_cols)
print("Numeric columns:", len(numeric_cols))
print("Top categorical cardinalities:")
print(cardinality.head(20))


try:
    ohe_low = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ohe_mid = OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=0.01,   # groups rare categories
        sparse_output=False
    )
except TypeError:
    ohe_low = OneHotEncoder(handle_unknown="ignore", sparse=False)
    ohe_mid = OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("yes_no_binary", Pipeline([("yes_no", YesNoBinaryEncoder())]), yes_no_cols_model),
        ("cat_ohe_low", Pipeline([("ohe", ohe_low)]), low_card_cols),
        ("cat_ohe_mid", Pipeline([("ohe", ohe_mid)]), mid_card_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
    sparse_threshold=0,
)


full_pipeline = Pipeline([("preprocessor", preprocessor)])


X_processed = full_pipeline.fit_transform(X_df)
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
processed_df = pd.DataFrame(X_processed, columns=feature_names, index=X_df.index)

print("Processed feature matrix shape:", processed_df.shape)
processed_df.head()

Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirli']
Low-cardinality categorical: []
Mid-cardinality categorical: ['address_part_1', 'address_part_2']
Numeric columns: 4
Top categorical cardinalities:
address_part_2    21
address_part_1    19
dtype: int64
Processed feature matrix shape: (1173, 36)


,yes_no_binary__has_document,yes_no_binary__avtodayanacaq,yes_no_binary__balkon,yes_no_binary__duzelme,yes_no_binary__esyali,yes_no_binary__hovuz,yes_no_binary__internet,yes_no_binary__isiq,yes_no_binary__kabel_tv,yes_no_binary__kombi,yes_no_binary__kondisioner,yes_no_binary__lift,yes_no_binary__merkezi_qizdirici_sistem,yes_no_binary__metbex_mebeli,yes_no_binary__pvc_pencere,yes_no_binary__qaz,yes_no_binary__su,yes_no_binary__telefon,yes_no_binary__temirli,cat_ohe_mid__address_part_1_Bakı,cat_ohe_mid__address_part_1_Sumqayıt,cat_ohe_mid__address_part_1_Xırdalan,cat_ohe_mid__address_part_1_infrequent_sklearn,cat_ohe_mid__address_part_2_ Abşeron,cat_ohe_mid__address_part_2_ Binəqədi,cat_ohe_mid__address_part_2_ Sabunçu,cat_ohe_mid__address_part_2_ Suraxanı,cat_ohe_mid__address_part_2_ Xətai,cat_ohe_mid__address_part_2_ Xəzər,cat_ohe_mid__address_part_2_ Yasamal,cat_ohe_mid__address_part_2_None,cat_ohe_mid__address_part_2_infrequent_sklearn,num__rooms,num__area_m2,num__land_area_sot,num__floor
0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,100.000000,1.7,1.0
1,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,110.000000,2.2,1.0
2,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,60.000000,0.6,1.0
3,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,250.000000,2.5,3.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0,204.700091,4.0,2.0


In [19]:
processed_df.to_csv('../../data/house/satilir_properties_house_processed.csv', index=False)
print('Preprocessed dataset saved to', '../../data/house/satilir_properties_house_preprocessed.csv')
y.to_csv('../../data/house/satilir_properties_house_target.csv', index=False)
print('Target variable saved to', '../../data/house/satilir_properties_house_target.csv')

Preprocessed dataset saved to ../../data/house/satilir_properties_house_preprocessed.csv
Target variable saved to ../../data/house/satilir_properties_house_target.csv
